# SIMG · Track HPC / Efficient ML · Sesión 01
# Introducción a Triton

**Notebook de trabajo de la sesión.** El andamiaje, los tests y el benchmark
están completos; **los kernels se escriben en vivo en las celdas vacías**.

> **Idea bisagra.** CUDA te hace programar un *thread escalar*; Triton te hace
> programar un *bloque que opera sobre un tile*. El compilador gestiona threads,
> memoria compartida, coalescencia y sincronización. Todo lo demás existe para aterrizar esto.

Arco: encuadre → fundamentos de GPU → modelo de programación → Kernel 1 (vector-add)
→ Kernel 2 (softmax fusionado + benchmark) → preview matmul → tooling.

Código de referencia completo: **`Referencia_codigo_Triton.pdf`** (en esta misma carpeta).

## 0 · Verificación del entorno

Triton necesita una **GPU NVIDIA**. En Colab: *Entorno de ejecución → Cambiar tipo → GPU (T4)*.
Si falla aquí, falla todo.

In [ ]:
import torch
import triton
import triton.language as tl

assert torch.cuda.is_available(), "Necesitas una GPU NVIDIA con CUDA."
print("PyTorch:", torch.__version__)
print("Triton :", triton.__version__)
print("GPU    :", torch.cuda.get_device_name(0))

## 1 · El encuadre: por qué Triton importa

La tensión es **PyTorch ↔ CUDA**. PyTorch es productivo pero te encierra en los ops
existentes y en los límites de fusión; CUDA da control total pero es doloroso. Triton
vive en el medio: sintaxis tipo Python, pero **tú escribes el kernel**.

- **FlashAttention se escribió en Triton.** Su ganancia está en *no materializar* la
  matriz de atención en HBM — la fusión memory-aware que Triton habilita.
- **El backend Inductor de `torch.compile` emite Triton.** Ya corres Triton aunque no
  lo escribas. Saber leerlo es saber qué hace tu modelo por dentro.

## 2 · Fundamentos de GPU (lo mínimo, no más)

- **Jerarquía de memoria:** HBM global (grande, lenta) → L2 → SRAM/shared (pequeña,
  rapidísima) → registros. **El juego es subir el dato a SRAM una vez y reusarlo.**
- **La mayoría de kernels de ML son *memory-bound*, no compute-bound.** El cuello de
  botella es el ancho de banda, no los FLOPs. Por eso fusionar es la palanca principal.
- **Roofline / intensidad aritmética** (FLOPs por byte): ubica si estás limitado por
  memoria o por cómputo.

## 3 · El modelo de programación de Triton

No escribes "qué hace el thread *i*", escribes "qué le pasa al tile que me tocó".

| Pieza | Para qué |
|---|---|
| `tl.program_id(axis)` | Qué bloque soy dentro del grid |
| **grid** | Cuántos bloques se lanzan; puede ser multidimensional |
| `tl.arange` + aritmética de punteros | Construir el vector de índices del tile |
| `mask` | Apagar lanes fuera de rango en los bordes |
| `tl.load` / `tl.store` con máscara | Único contacto explícito con memoria |
| `constexpr` (`BLOCK_SIZE`) | Valor conocido en compilación; especializa el kernel |
| `num_warps`, `num_stages`, `@triton.autotune` | Perillas de rendimiento, se *miden* |

## 4 · Kernel 1 — Suma de vectores

El "hola mundo". Enseña la mecánica mínima: **grid, `program_id`, índices, máscara,
load/store**. Escríbelo en la celda vacía (referencia en el PDF).

Pasos: ¿qué program soy? → offsets del bloque → máscara → load → suma → store.

**Wrapper** (andamiaje). El `grid` es función de los meta-parámetros, por eso `cdiv`.

In [ ]:
def add(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    output = torch.empty_like(x)
    assert x.is_cuda and y.is_cuda and output.is_cuda
    n_elements = output.numel()
    grid = lambda meta: (triton.cdiv(n_elements, meta["BLOCK_SIZE"]),)
    add_kernel[grid](x, y, output, n_elements, BLOCK_SIZE=1024)
    return output

**Verificación** — debe imprimir `OK ✓` y diferencia máxima `0`.

In [ ]:
torch.manual_seed(0)
size = 98_432  # a propósito NO múltiplo de 1024 -> prueba la máscara
x = torch.rand(size, device="cuda")
y = torch.rand(size, device="cuda")

out_triton = add(x, y)
out_torch = x + y
max_diff = (out_triton - out_torch).abs().max().item()
print(f"Diferencia máxima Triton vs PyTorch: {max_diff:.3e}")
assert max_diff == 0, "¡Los resultados no coinciden!"
print("OK ✓  La suma de vectores en Triton coincide con PyTorch.")

> **No vendas esto de más.** Vector-add es memory-bound trivial: lee dos arrays,
> escribe uno. No demuestra el valor de Triton. Es andamiaje — pasa rápido al softmax.

## 5 · Kernel 2 — Softmax fusionado (el "ajá")

El softmax *naïve* hace varias pasadas por HBM (max, resta, exp, suma, división). El
fusionado **carga la fila una vez a SRAM, hace todo ahí, y escribe una vez**. El número
del benchmark es lo que convence.

Suposición didáctica: cada fila cabe en un bloque (`BLOCK_SIZE >= n_cols`). Recuerda el
relleno `-inf` (neutro del max y del exp). Escríbelo en la celda vacía.

**Wrapper** (andamiaje). `BLOCK_SIZE` = siguiente potencia de 2 ≥ `n_cols`.

In [ ]:
def softmax(x: torch.Tensor) -> torch.Tensor:
    n_rows, n_cols = x.shape
    BLOCK_SIZE = triton.next_power_of_2(n_cols)
    out = torch.empty_like(x)
    softmax_kernel[(n_rows,)](
        out, x,
        x.stride(0), out.stride(0),
        n_cols,
        BLOCK_SIZE=BLOCK_SIZE,
    )
    return out

**Verificación** — `torch.allclose` contra `torch.softmax`.

In [ ]:
torch.manual_seed(0)
x = torch.randn(1823, 781, device="cuda")
out_triton = softmax(x)
out_torch = torch.softmax(x, axis=1)
max_diff = (out_triton - out_torch).abs().max().item()
print(f"Diferencia máxima Triton vs PyTorch: {max_diff:.3e}")
assert torch.allclose(out_triton, out_torch, atol=1e-6), "¡No coinciden!"
print("OK ✓  El softmax fusionado coincide con PyTorch.")

### Benchmark — el número que convence

`do_bench` hace **warmup y repeticiones**, excluyendo el costo de compilación de la
primera llamada. Sin esto los números mienten — y un experto lo nota.

In [ ]:
@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=["N"],
        x_vals=[256 * i for i in range(2, 50)],
        line_arg="provider",
        line_vals=["triton", "torch"],
        line_names=["Triton", "PyTorch"],
        ylabel="GB/s",
        plot_name="softmax",
        args={"M": 4096},
    )
)
def bench(M, N, provider):
    x = torch.randn(M, N, device="cuda")
    if provider == "triton":
        fn = lambda: softmax(x)
    else:
        fn = lambda: torch.softmax(x, axis=-1)
    ms = triton.testing.do_bench(fn)
    return 2 * x.numel() * x.element_size() / ms * 1e-6   # GB/s

bench.run(print_data=True, show_plots=True)

## 6 · Preview — Matmul + autotuning

*No se enseña a fondo hoy: es su propia sesión.* Tres ideas a **teasear**: acumulación
por tiles a lo largo de K con acumulador en **fp32**, **ordenamiento agrupado** de
programas (reuso de L2), y que **`tl.dot` usa Tensor Cores**. El autotuner re-tunea
cuando cambian las formas `(M, N, K)`. (Referencia completa en `../kernels/03_matmul.py`.)

In [ ]:
@triton.autotune(
    configs=[
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 256, "BLOCK_K": 64, "GROUP_M": 8},
                      num_stages=3, num_warps=8),
        triton.Config({"BLOCK_M": 64, "BLOCK_N": 64, "BLOCK_K": 32, "GROUP_M": 8},
                      num_stages=4, num_warps=4),
    ],
    key=["M", "N", "K"],
)
@triton.jit
def matmul_kernel(
    a_ptr, b_ptr, c_ptr, M, N, K,
    stride_am, stride_ak, stride_bk, stride_bn, stride_cm, stride_cn,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
    GROUP_M: tl.constexpr,
):
    pid = tl.program_id(0)
    n_m = tl.cdiv(M, BLOCK_M)
    n_n = tl.cdiv(N, BLOCK_N)
    per_group = GROUP_M * n_n
    gid = pid // per_group
    first_m = gid * GROUP_M
    gsize = min(n_m - first_m, GROUP_M)
    pid_m = first_m + (pid % gsize)
    pid_n = (pid % per_group) // gsize

    offs_m = (pid_m * BLOCK_M + tl.arange(0, BLOCK_M)) % M
    offs_n = (pid_n * BLOCK_N + tl.arange(0, BLOCK_N)) % N
    offs_k = tl.arange(0, BLOCK_K)
    a_ptrs = a_ptr + offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak
    b_ptrs = b_ptr + offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_K)):
        a = tl.load(a_ptrs, mask=offs_k[None, :] < K - k * BLOCK_K, other=0.0)
        b = tl.load(b_ptrs, mask=offs_k[:, None] < K - k * BLOCK_K, other=0.0)
        acc += tl.dot(a, b)
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk

    offs_cm = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_cn = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    c_ptrs = c_ptr + offs_cm[:, None] * stride_cm + offs_cn[None, :] * stride_cn
    c_mask = (offs_cm[:, None] < M) & (offs_cn[None, :] < N)
    tl.store(c_ptrs, acc.to(tl.float16), mask=c_mask)


def matmul(a, b):
    M, K = a.shape
    K, N = b.shape
    c = torch.empty((M, N), device=a.device, dtype=torch.float16)
    grid = lambda META: (triton.cdiv(M, META["BLOCK_M"]) * triton.cdiv(N, META["BLOCK_N"]),)
    matmul_kernel[grid](
        a, b, c, M, N, K,
        a.stride(0), a.stride(1), b.stride(0), b.stride(1), c.stride(0), c.stride(1),
    )
    return c


a = torch.randn((512, 512), device="cuda", dtype=torch.float16)
b = torch.randn((512, 512), device="cuda", dtype=torch.float16)
c_triton = matmul(a, b)
c_torch = torch.matmul(a, b)
print("Diferencia máxima:", (c_triton - c_torch).abs().max().item())
assert torch.allclose(c_triton, c_torch, atol=1e-1, rtol=1e-1)
print("OK ✓  Matmul por bloques con autotuning. (Esto es la próxima sesión.)")

## 7 · Tooling y entorno

### Debugging — el truco que más agradece la sala

`TRITON_INTERPRET=1` ejecuta el **mismo** kernel en CPU con semántica Python: funcionan
`print()` y `pdb` dentro del kernel, y los vectores se vuelven arrays de numpy.

```bash
TRITON_INTERPRET=1 python mi_script.py
```

En notebook: ponla **antes** de importar Triton y reinicia el runtime.

### Autograd — Triton NO autodiferencia

Escribes forward y backward **a mano** y los envuelves en un `torch.autograd.Function`.
Es deliberado: control sobre qué se recomputa vs. qué se guarda (la idea de FlashAttention).

In [ ]:
class _FusedSoftmax(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        y = softmax(x)                 # nuestro kernel Triton del bloque 5
        ctx.save_for_backward(y)
        return y

    @staticmethod
    def backward(ctx, dy):
        (y,) = ctx.saved_tensors
        s = (dy * y).sum(axis=-1, keepdim=True)
        return y * (dy - s)


fused_softmax = _FusedSoftmax.apply

xa = torch.randn(64, 128, device="cuda", requires_grad=True)
xb = xa.detach().clone().requires_grad_(True)
fused_softmax(xa).sum().backward()
torch.softmax(xb, axis=-1).sum().backward()
print("grad max diff:", (xa.grad - xb.grad).abs().max().item())

## Cierre

Una sola cosa: **CUDA = thread escalar; Triton = bloque sobre un tile.** El resto
(grid, máscara, fusión, autotuning) aterriza eso.

**Antes de la próxima sesión:** corre los tres kernels contra *tu* versión de Triton y
guarda los números del benchmark del softmax en *tu* hardware.